In [5]:
!pip install transformers albumentations tqdm

In [6]:
import os
import cv2
import torch
import numpy as np
import albumentations as A

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation

import torch.nn.functional as F

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
mapping = {
100:0,
200:1,
300:2,
550:3,
600:4,
700:5,
800:6,
7100:7,
10000:8
}

NUM_CLASSES = 9

In [9]:
train_transform = A.Compose([

    A.Resize(512,512),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    A.RandomRotate90(p=0.5),

    A.Affine(
        translate_percent=0.1,
        scale=(0.8,1.2),
        rotate=(-25,25),
        p=0.7
    ),

    A.GaussianBlur(p=0.2),
    A.RandomBrightnessContrast(p=0.4),

])

val_transform = A.Compose([
    A.Resize(512,512)
])

In [10]:
class DesertDataset(Dataset):

    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(image_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        # Read image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # IMPORTANT: read mask as single channel
        raw_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Convert dataset labels → class IDs
        mask = np.zeros(raw_mask.shape, dtype=np.uint8)

        for k, v in mapping.items():
            mask[raw_mask == k] = v

        # Apply augmentation
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask = aug["mask"]

        # Convert to tensor
        image = torch.tensor(image).permute(2,0,1).float() / 255.0
        mask = torch.tensor(mask).long()

        return image, mask

In [11]:
train_dataset = DesertDataset(
"/kaggle/input/datasets/mohith2045/desert-segmentation/Offroad_Segmentation_Training_Dataset/train/Color_Images",
"/kaggle/input/datasets/mohith2045/desert-segmentation/Offroad_Segmentation_Training_Dataset/train/Color_Images",
transform=train_transform
)

val_dataset = DesertDataset(
"/kaggle/input/datasets/mohith2045/desert-segmentation/Offroad_Segmentation_Training_Dataset/val/Color_Images",
"/kaggle/input/datasets/mohith2045/desert-segmentation/Offroad_Segmentation_Training_Dataset/val/Segmentation",
transform=val_transform
)

In [12]:
train_loader = DataLoader(
train_dataset,
batch_size=2,
shuffle=True,
num_workers=2
)

val_loader = DataLoader(
val_dataset,
batch_size=2,
shuffle=False,
num_workers=2
)

In [13]:
model = SegformerForSemanticSegmentation.from_pretrained(
"nvidia/segformer-b4-finetuned-ade-512-512",
num_labels=NUM_CLASSES,
ignore_mismatched_sizes=True
)

model.to(device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/257M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/930 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b4-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([9])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([9, 768, 1, 1])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [14]:
def dice_loss(pred, target):

    pred = F.softmax(pred, dim=1)

    target_onehot = F.one_hot(target, NUM_CLASSES).permute(0,3,1,2)

    intersection = (pred * target_onehot).sum()
    union = pred.sum() + target_onehot.sum()

    dice = (2 * intersection + 1) / (union + 1)

    return 1 - dice

In [15]:
ce_loss = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
model.parameters(),
lr=3e-5,
weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
optimizer,
T_max=30
)

In [16]:
scaler = torch.amp.GradScaler("cuda")

In [17]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for images, masks in tqdm(train_loader):

        images = images.to(device)
        masks = masks.to(device)

        with torch.amp.autocast("cuda"):

            outputs = model(pixel_values=images).logits

            outputs = F.interpolate(
                outputs,
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            loss = ce_loss(outputs, masks) + dice_loss(outputs, masks)

        optimizer.zero_grad()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    scheduler.step()

    print("Epoch",epoch+1,"Loss:",total_loss/len(train_loader))

  0%|          | 0/1429 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/257M [00:00<?, ?B/s]

100%|██████████| 1429/1429 [06:55<00:00,  3.44it/s]


Epoch 1 Loss: 0.45876950360547836


100%|██████████| 1429/1429 [06:53<00:00,  3.45it/s]


...

Epoch 8 Loss: 0.023602120525712728


100%|██████████| 1429/1429 [06:52<00:00,  3.46it/s]


Epoch 9 Loss: 0.02288472749069083


100%|██████████| 1429/1429 [06:52<00:00,  3.47it/s]

Epoch 10 Loss: 0.02261513023918406


In [18]:
def compute_iou(pred, mask):

    pred = pred.view(-1)
    mask = mask.view(-1)

    ious = []

    for cls in range(NUM_CLASSES):

        pred_inds = pred == cls
        target_inds = mask == cls

        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()

        if union == 0:
            continue

        ious.append(intersection/union)

    return np.mean(ious)

In [19]:
model.eval()

total_iou = 0
count = 0

with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(pixel_values=images).logits

        outputs = F.interpolate(
            outputs,
            size=masks.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        preds = outputs.argmax(1)

        iou = compute_iou(preds,masks)

        total_iou += iou
        count += 1

print("Mean IoU:",total_iou/count)

Mean IoU: 0.9244661631074341


In [20]:
torch.save(model.state_dict(),"segformer_b4_0.9iou_model.pth")

In [22]:
test_dataset = DesertDataset(
    "/kaggle/input/datasets/mohith2045/desert-test/Offroad_Segmentation_testImages/Color_Images",
    "/kaggle/input/datasets/mohith2045/desert-test/Offroad_Segmentation_testImages/Segmentation",
    transform=val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

In [23]:
model.eval()

total_iou = 0
count = 0

with torch.no_grad():

    for images, masks in test_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(pixel_values=images).logits

        outputs = F.interpolate(
            outputs,
            size=masks.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        preds = outputs.argmax(1)

        iou = compute_iou(preds, masks)

        total_iou += iou
        count += 1

final_iou = total_iou / count

print("Test Dataset Mean IoU:", final_iou)

Test Dataset Mean IoU: 0.9990019579371531
